[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_5_advanced_analytics/assignments/multitemporal_analysis.ipynb)

# Assignment 5B: Analysis of multitemporal UAS data

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

In this notebook we analyze the seven UAS DSMs of Mid Pines flown between March and October 2015 as a time series. We register them as a space-time raster dataset in the GRASS temporal framework, find the area covered by every survey, and aggregate the series by month. We derive the core surface, compare it with the lidar bare ground to find the two distorted surveys, and recompute the core without them. We then apply the series: crop height and biomass from map algebra on the series, a webcam viewshed on the lidar DSM and on the envelope DSM, and the volume of a building.

Keep in mind that the seven flights were processed with different numbers of GCPs, from two to eleven. The surveys with few GCPs are the ones the core surface exposes. Parts 3 to 5 are computed inside the intersection of the surveys and under a mask, so their statistics refer to that area only.

This notebook runs the same steps as the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_5_advanced_analytics/assignments/assignment_5b.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed. The Timeline Tool and the Temporal Plot Tool of the GUI are replaced here by matplotlib plots.

## Learning objectives

- Register a series of DSMs as a space-time raster dataset and query it by time
- Derive the intersection of the surveys, monthly aggregates, and per-cell statistics over time
- Use the core surface and the time of minimum to find distorted surveys
- Apply map algebra on the series to derive crop height and biomass
- Compare viewsheds on the lidar DSM and on the envelope DSM, and compute the volume of a structure

## Outline

- Part 1: Environment setup
- Part 2: Register the time series
- Part 3: Intersection and aggregation
- Part 4: Core surface and distorted surveys
- Part 5: Crop height and biomass
- Part 6: Viewsheds and the envelope DSM
- Part 7: Volume of a building
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5). We also import pandas and matplotlib for the tables and plots.

In [ ]:
import io
import os
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB); its `PERMANENT` mapset holds the seven 2015 DSMs
- The 2013 lidar DEM and DSM of Mid Pines as GRASS raster packs (zip, about 90 MB)
- The fields raster as a GRASS raster pack, the series file with the DSM names and dates, and the color rules for difference maps

The downloads total about 320 MB and take a minute or two on Colab.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
LIDAR_URL = f"{BUCKET}/lidar-data/Midpines/secref_mid_pines_spm_elev.zip"
FIELDS_URL = f"{BUCKET}/grassdata/fields.pack"
SERIES_URL = f"{BUCKET}/grassdata/series.txt"
COLORS_URL = f"{BUCKET}/grassdata/dif_lidar_uav.txt"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o secref_mid_pines_spm_elev.zip {LIDAR_URL} && unzip -q -o secref_mid_pines_spm_elev.zip
!curl -sSL -o fields.pack {FIELDS_URL}
!curl -sSL -o series.txt {SERIES_URL}
!curl -sSL -o dif_lidar_uav.txt {COLORS_URL}
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `multitemporal` inside the Lake Wheeler project for our work (the `PERMANENT` mapset stays untouched), start the session there, and create a `Tools` object. From here on, `tools.t_create(...)` runs `t.create`, `tools.g_region(...)` runs `g.region`, and so on; a flag like `-n` is passed as `flags="n"`.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "multitemporal"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session)

print(tools.g_proj(flags="p").text)

### Helpers

`describe` prints the statistics of a raster in the current region from `r.univar`. `table` turns the pipe-separated output of the `t.*` listing tools into a pandas table. `show` draws a raster in the current region with a legend.

In [ ]:
def describe(raster, label=None):
    """Print the range, mean, and spread of a raster in the current region."""
    s = tools.r_univar(map=raster, flags="e", format="json").json
    print(f"{label or raster}: {s['n']:,} cells")
    print(f"  min {s['min']:8.2f}  max {s['max']:8.2f}  mean {s['mean']:8.2f}  median {s['median']:8.2f}  stddev {s['stddev']:6.2f}")
    return s


def table(text):
    """Read the pipe-separated output of a t.* listing tool into a DataFrame."""
    return pd.read_csv(io.StringIO(text), sep="|")


def show(raster, title=None, legend=None, width=500, height=500, flags="sb", shade=None):
    """Draw one raster in the current region, optionally draped over a shaded relief."""
    m = gj.Map(width=width, height=height, use_region=True)
    if shade:
        m.d_shade(shade=shade, color=raster)
    else:
        m.d_rast(map=raster)
    if legend:
        m.d_legend(raster=raster, at=(5, 45, 3, 6), title=legend, flags=flags, bgcolor="white")
    m.d_barscale(at=(55, 5))
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()
    return m

---
## Part 2: Register the time series

### Copy the DSMs

The temporal database lives in the mapset, and maps from other mapsets cannot be registered in it, so we copy the seven DSMs from `PERMANENT`. The names record the flight date and the number of GCPs used in processing.

In [ ]:
DSMS = [
    "2015_03_18_DSM_agi_6GCP",
    "2015_06_20_DSM_agi_11GCP",
    "2015_07_27_DSM_agi_2GCP_4ID",
    "2015_08_14_DSM_agi_3GCP_4ID",
    "2015_09_04_DSM_agi_6GCP",
    "2015_09_21_DSM_agi_6GCP",
    "2015_10_06_DSM_agi_8GCPs",
]
for name in DSMS:
    tools.g_copy(raster=f"{name}@PERMANENT,{name}", overwrite=True)
print(tools.g_list(type="raster", pattern="2015*", mapset=".").text)

### Create and register the dataset

We create a space-time raster dataset (STRDS) with absolute time and register the seven maps from `series.txt`, which lists each DSM with its date, one per line, separated by a pipe.

In [ ]:
print(Path("series.txt").read_text())

tools.t_create(output="uas_dsm", type="strds", temporaltype="absolute", semantictype="mean",
               title="UAS_DSM", description="UAS-based 2015 DSMs processed by Agisoft", overwrite=True)
tools.t_register(input="uas_dsm", file="series.txt")
print(tools.t_info(input="uas_dsm").text)

`t.rast.list` lists the registered maps; the `where` clause is SQL on the time columns, and `columns` can include statistics of each map.

In [ ]:
print(tools.t_rast_list(input="uas_dsm", columns="name,start_time", where="start_time >= '2015-08-01'").text)
series = table(tools.t_rast_list(input="uas_dsm", columns="name,start_time,min,max").text)
series["start_time"] = pd.to_datetime(series["start_time"])
series

#### Plot: the timeline of the series

The GUI Timeline Tool shows when each map was taken; here the same thing with matplotlib.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 1.8))
ax.plot(series["start_time"], [1] * len(series), "o")
for _, row in series.iterrows():
    ax.annotate(row["start_time"].strftime("%b %d"), (row["start_time"], 1), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_yticks([])
ax.set_xlabel("2015")
plt.tight_layout()
plt.show()

#### Map: two of the DSMs with a shared color table

`t.rast.colors` sets one color table for every map in the dataset.

In [ ]:
tools.t_rast_colors(input="uas_dsm", color="elevation")
tools.g_region(raster="2015_06_20_DSM_agi_11GCP")
for name in ["2015_06_20_DSM_agi_11GCP", "2015_10_06_DSM_agi_8GCPs"]:
    show(name, title=name, legend="Elevation (m)", width=460, height=460)

#### Plot: elevation through time at three points

The GUI Temporal Plot Tool plots the value of every map at one location. We sample each map with `r.what` at a point on the road, in a field, and on the building.

In [ ]:
POINTS = {"road": (637080, 219480), "field": (636879, 219432), "building": (637120, 219376)}

rows = []
for name, date in zip(series["name"], series["start_time"]):
    for label, (x, y) in POINTS.items():
        value = tools.r_what(map=name, coordinates=(x, y), format="json").json[0][name]["value"]
        rows.append({"date": date, "point": label, "elevation": value})
tplot = pd.DataFrame(rows).pivot(index="date", columns="point", values="elevation")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, label in zip(axes, POINTS):
    ax.plot(tplot.index, tplot[label], "o-")
    ax.set_title(label)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Elevation (m)")
plt.tight_layout()
plt.show()
tplot.round(2)

> **Think about it:** A road should stay flat and a field should rise through the season and drop at harvest. Which surveys break that pattern, and at which points?

---
## Part 3: Intersection and aggregation

### Area mapped by all surveys

Per-cell statistics over the series are only meaningful where every survey has data. We set the region to the union of the extents, which `t.info -g` reports, and count the maps with data in each cell with `t.rast.series`. With the `-n` flag the count is null wherever any map is null, which gives the intersection.

In [ ]:
info = tools.t_info(input="uas_dsm", flags="g").keyval
print({k: info[k] for k in ["north", "south", "east", "west", "nsres_min"]})

tools.g_region(n=219942, s=219037, e=637439, w=636456, res=0.3, flags="pa")
tools.t_rast_series(input="uas_dsm", method="count", output="count", overwrite=True)
tools.t_rast_series(input="uas_dsm", method="count", output="intersection", flags="n", overwrite=True)
tools.r_colors(map="count", color="viridis")
show("count", title="Number of surveys with data", legend="Surveys", width=460, height=460)
show("intersection", title="Intersection of all surveys", width=460, height=460)

We shrink the region to the intersection and set it as the mask for Parts 3 to 5. Every raster command from here on reads and writes only inside the mask.

In [ ]:
tools.g_region(zoom="intersection", flags="p")
tools.r_mask(raster="intersection", overwrite=True)
print(tools.r_report(map="intersection", units="h", flags="n").text)

### Monthly aggregation

The dataset has two surveys in September. `t.rast.aggregate` combines the maps that fall in each interval of a chosen granularity into one map, here the monthly average, and registers the results as a new dataset with interval time stamps.

In [ ]:
tools.t_rast_aggregate(input="uas_dsm", output="uas_dsm_aggr", basename="uas_dsm_aggr", suffix="time",
                       granularity="1 months", method="average", overwrite=True)
table(tools.t_rast_list(input="uas_dsm_aggr", columns="name,start_time,end_time").text)

> **Think about it:** Six monthly maps result from seven surveys. Which month's map is an average of two surveys, and what does averaging do to the crop rows if the two surveys are not perfectly aligned?

---
## Part 4: Core surface and distorted surveys

### Minimum elevation

We restore the lidar surfaces, compute the core surface, the minimum elevation of the series at each cell, and subtract it from the lidar bare ground. A DSM cannot lie below the ground it was flown over, so a core surface well below the lidar DEM points at a distorted survey.

In [ ]:
tools.r_unpack(input="mid_pines_lidar2013_dsm.pack", flags="o", overwrite=True)
tools.r_unpack(input="mid_pines_lidar2013_dem.pack", flags="o", overwrite=True)

tools.t_rast_series(input="uas_dsm", method="minimum", output="minimum", overwrite=True)
tools.r_mapcalc(expression="diff_lidar_uas = mid_pines_lidar2013_dem - minimum", overwrite=True)
tools.r_colors(map="diff_lidar_uas", color="differences")
_ = describe("diff_lidar_uas", "Lidar ground minus core, all surveys")
show("diff_lidar_uas", title="Lidar ground minus core, all surveys", legend="Difference (m)")

### Which survey is responsible

`method=min_raster` writes, for each cell, the index of the map that supplied the minimum, counted from 0 in the order of the time stamps. We label the indexes with the flight dates so the legend reads them directly.

In [ ]:
tools.t_rast_series(input="uas_dsm", method="min_raster", output="min_raster", overwrite=True)
labels = "\n".join(f"{i}:{d.strftime('%Y-%m-%d')}" for i, d in enumerate(series["start_time"]))
gs.write_command("r.category", map="min_raster", rules="-", stdin=labels + "\n", separator=":", quiet=True)
tools.r_colors(map="min_raster", color="viridis")
show("min_raster", title="Survey with the minimum elevation", legend="Survey", flags="b")

for x, y in [(636879, 219432), (637232, 219651)]:
    print(tools.r_what(map="min_raster", coordinates=(x, y), flags="f").text.strip())
print(tools.r_report(map="min_raster", units="p", flags="n").text)

### Recompute the core without them

Index 2 is the third map, the July 27 survey, and index 3 the fourth, August 14; together they supply the minimum in two thirds of the cells. These two flights were processed with only two and three GCPs. We exclude them with a `where` clause on the time stamps and recompute the core and the difference, this time with the color rules from Assignment 5A: grey within 10 cm, color beyond.

In [ ]:
EXCLUDE = "start_time < '2015-07-01' or start_time > '2015-09-01'"

tools.t_rast_series(input="uas_dsm", method="minimum", output="minimum", where=EXCLUDE, overwrite=True)
tools.r_mapcalc(expression="diff_lidar_uas = mid_pines_lidar2013_dem - minimum", overwrite=True)
_ = describe("diff_lidar_uas", "Lidar ground minus core, distorted surveys excluded")
tools.r_colors(map="diff_lidar_uas", rules="dif_lidar_uav.txt")
show("diff_lidar_uas", title="Lidar ground minus core, five surveys", legend="Difference (m)")
print(tools.t_list(columns="id,number_of_maps,start_time").text)

> **Think about it:** The mean difference drops from 0.87 m to 0.08 m. What remains colored in the second map, and why?

---
## Part 5: Crop height and biomass

Map algebra on a series applies one expression to every map and registers the results as a new dataset. We compute the height of the UAS surface above the lidar bare ground, keep it only inside the crop fields and between 0.3 and 2 m, to drop the soil noise below and the building and trees above, and set the rest to null. In the expression, `uas_dsm` stands for each registered map in turn.

In [ ]:
tools.r_unpack(input="fields.pack", flags="o", overwrite=True)
tools.t_rast_mapcalc(
    inputs="uas_dsm",
    expression="if (fields, if(uas_dsm - mid_pines_lidar2013_dem > 0.3 && uas_dsm - mid_pines_lidar2013_dem < 2, uas_dsm - mid_pines_lidar2013_dem, null()))",
    output="veg_uas_lidar", basename="veg_uas_lidar", overwrite=True,
)
table(tools.t_rast_list(input="veg_uas_lidar", columns="name,start_time").text)

The outputs are named `veg_uas_lidar_1` to `veg_uas_lidar_7`, numbered from 1 in time order, so `veg_uas_lidar_2` is June 20. `t.rast.univar` summarizes every map of the series; we leave out the two distorted surveys and compute the crop area and volume per survey from the cell count and the sum of heights.

In [ ]:
region = tools.g_region(flags="g", format="json").json
cell_area = region["nsres"] * region["ewres"]

veg = table(tools.t_rast_univar(input="veg_uas_lidar", where=EXCLUDE).text)
veg["date"] = pd.to_datetime(veg["start"])
veg["crop area (ha)"] = veg["non_null_cells"] * cell_area / 10000
veg["crop volume (m3)"] = veg["sum"] * cell_area
veg[["id", "date", "mean", "crop area (ha)", "crop volume (m3)"]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(veg["date"], veg["crop volume (m3)"], width=8)
ax.set_ylabel("Crop volume above ground (m$^3$)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

#### Map: crop height on June 20

In [ ]:
tools.r_colors(map="veg_uas_lidar_2", color="viridis")
show("veg_uas_lidar_2", title="Crop height above lidar ground, 2015-06-20", legend="Height (m)")
print(tools.r_report(map="veg_uas_lidar_2", units="me", flags="n").text)

The crop volume above ground is the sum of the heights times the cell area, which is what the assignment page computes in the GUI Python shell.

In [ ]:
stats = tools.r_univar(map="veg_uas_lidar_2", format="json").json
volume = stats["sum"] * cell_area
print(f"June 20: {stats['n']:,} crop cells, {stats['n'] * cell_area:,.0f} m2, volume {volume:,.0f} m3")

# Remove the mask before Part 6, which works on the whole study area.
tools.r_mask(flags="r")

> **Think about it:** How does the crop volume change from March to October? Does the October value mean the fields are bare?

---
## Part 6: Viewsheds and the envelope DSM

### Cumulative viewshed

We want to place a webcam that watches the fields. First we compute a cumulative viewshed from six candidate points on the lidar DSM, with the camera 3 m above the ground, at 1 m resolution. Each viewshed holds the vertical angle to the visible cells and null elsewhere; `r.series` with `method=count` counts, for every cell, from how many points it is visible.

In [ ]:
CANDIDATES = [(637087, 219376), (637004, 219653), (636782, 219374), (636839, 219150), (636665, 219527), (636863, 219503)]

tools.g_region(raster="mid_pines_lidar2013_dsm", res=1, flags="pa")
for i, (x, y) in enumerate(CANDIDATES, start=1):
    tools.r_viewshed(input="mid_pines_lidar2013_dsm", output=f"vshed_{i}", coordinates=(x, y), observer_elevation=3, overwrite=True)
tools.r_series(input=",".join(f"vshed_{i}" for i in range(1, 7)), output="cum_viewshed", method="count", overwrite=True)
tools.r_colors(map="cum_viewshed", color="viridis")

gs.write_command("v.in.ascii", input="-", output="candidates", separator="comma", overwrite=True,
                 stdin="\n".join(f"{x},{y}" for x, y in CANDIDATES) + "\n", quiet=True)
m = gj.Map(width=500, height=500, use_region=True)
m.d_rast(map="cum_viewshed")
m.d_vect(map="candidates", icon="basic/circle", size=10, fill_color="red", color="black")
m.d_legend(raster="cum_viewshed", at=(5, 45, 3, 6), title="Visible from N points", flags="b", bgcolor="white")
m.d_barscale(at=(55, 5))
m.show()
print(tools.r_report(map="cum_viewshed", units="p", flags="n").text)

### Envelope DSM

The camera has to see the fields through the whole season, so the test surface is the envelope of the series, the maximum elevation at each cell, fused into the lidar DSM inside the fields. We compute the envelope without the two distorted surveys at the native resolution, clip it to the fields, and fuse it with `r.patch.smooth` as in Assignment 5A. The addon is installed with `g.extension`; addons installed after the `Tools` object was created are called with `grass.script`.

In [ ]:
tools.g_extension(extension="r.patch.smooth")

In [ ]:
tools.g_region(raster="mid_pines_lidar2013_dsm", flags="p")
tools.t_rast_series(input="uas_dsm", method="maximum", output="maximum", where=EXCLUDE, overwrite=True)
tools.r_mapcalc(expression="maximum_clip = if(fields, maximum)", overwrite=True)
gs.run_command("r.patch.smooth", input_a="maximum_clip", input_b="mid_pines_lidar2013_dsm", output="lidar_maximum_dsm", smooth_dist=10, overwrite=True)
tools.r_relief(input="lidar_maximum_dsm", output="lidar_maximum_dsm_relief", zscale=5, overwrite=True)
tools.r_relief(input="mid_pines_lidar2013_dsm", output="mid_pines_lidar2013_dsm_relief", zscale=5, overwrite=True)

#### Map: the envelope DSM around the building

In [ ]:
tools.g_region(n=219440, s=219320, w=637040, e=637200, res=0.3)
show("mid_pines_lidar2013_dsm_relief", title="Lidar DSM 2013", width=420, height=320)
show("lidar_maximum_dsm_relief", title="Envelope DSM, five surveys of 2015", width=420, height=320)

> **Think about it:** The envelope keeps everything that was ever there during a survey. What do the rows of bumps along the road beside the building show?

### Compare the viewsheds

We pick a camera location from the cumulative viewshed, here a point with a wide view, and compute its viewshed on the lidar DSM and on the envelope DSM at 1 m resolution.

In [ ]:
CAMERA = (636917, 219223)

tools.g_region(raster="mid_pines_lidar2013_dsm", res=1, flags="pa")
tools.r_viewshed(input="mid_pines_lidar2013_dsm", output="viewshed_lidar_dsm", coordinates=CAMERA, observer_elevation=3, overwrite=True)
tools.r_viewshed(input="lidar_maximum_dsm", output="viewshed_maximum", coordinates=CAMERA, observer_elevation=3, overwrite=True)
tools.r_colors(map="viewshed_lidar_dsm", color="reds")
tools.r_colors(map="viewshed_maximum", color="greens")

for name in ["viewshed_lidar_dsm", "viewshed_maximum"]:
    n = tools.r_univar(map=name, format="json").json["n"]
    print(f"{name}: {n:,} visible cells, {n / 10000:.1f} ha")

gs.write_command("v.in.ascii", input="-", output="camera", separator="comma", stdin=f"{CAMERA[0]},{CAMERA[1]}\n", overwrite=True, quiet=True)
m = gj.Map(width=500, height=500, use_region=True)
m.d_rast(map="mid_pines_lidar2013_dsm_relief")
m.d_rast(map="viewshed_lidar_dsm")
m.d_rast(map="viewshed_maximum")
m.d_vect(map="camera", icon="basic/star", size=14, fill_color="yellow", color="black")
m.d_barscale(at=(55, 5))
m.show()

Red is visible only on the lidar DSM, green on the envelope DSM as well, since the green map is drawn on top.

> **Think about it:** Where does the crop close the view? Would a higher camera or another candidate point do better?

---
## Part 7: Volume of a building

The difference between a DSM and the bare ground over the footprint of a feature gives its volume. We set the region to the building, compute the height of the June DSM above the lidar DEM, take everything more than 2 m high as the footprint, and let `r.volume` sum the height over the clump and multiply by the cell area.

In [ ]:
tools.g_region(n=219388.5, s=219366.9, w=637100.1, e=637138.5, res=0.3, flags="pa")
tools.r_mapcalc(expression="building = 2015_06_20_DSM_agi_11GCP - mid_pines_lidar2013_dem", overwrite=True)
tools.r_mapcalc(expression="building_mask = if(building > 2, 1, null())", overwrite=True)
print(tools.r_volume(input="building", clump="building_mask").text)
show("building", legend="Height (m)", width=520, height=310)

> **Think about it:** The footprint is defined by a 2 m threshold on the height. What does the threshold do at the walls, and which input limits the accuracy of the volume more: the UAS DSM or the lidar DEM?

---
## Report questions

Answer each question in the Markdown cell below it. These answers, with the maps from this notebook, are the content of your report (see the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_5_advanced_analytics/assignments/assignment_5b.html) for the submission details).

#### Question 1: Series and intersection

Describe the registered dataset, the count and intersection maps, and the intersection area. What do the elevation plots at the road, field, and building points show?

`Add text here with your answer to question 1`

#### Question 2: Aggregation

List the monthly maps. What does the September aggregate represent, and what are the consequences of averaging two surveys?

`Add text here with your answer to question 2`

#### Question 3: Core surface

Compare the lidar minus core maps before and after excluding the distorted surveys, with their means. How did you identify the two surveys, and why are they distorted?

`Add text here with your answer to question 3`

#### Question 4: Crop height and biomass

Report the June crop area and volume and the table for the valid surveys. How does the crop change through the season?

`Add text here with your answer to question 4`

#### Question 5: Viewsheds

Describe the cumulative viewshed and the two viewsheds at the camera point with their areas. How does the crop change the view, and where would you put the camera?

`Add text here with your answer to question 5`

#### Question 6: Building volume

Report the volume, how the footprint was defined, and what limits the accuracy of the estimate.

`Add text here with your answer to question 6`